# Ariketa 5.2 — Sare baten anatomia Keras-en 🧱 (EGITEKO)

S1ean sare bat **eskuz** kodetu genuen. Gaur Keras-ek forward-a, gradienteen kalkulua (**backprop / autodiff**) eta pisuen eguneraketa automatizatuko ditu. Gako-ideia: **Keras ez da magia** — pieza bakoitza S1eko kontzeptu baten abstrakzioa da.


> 🌐 **5072 ML LAB · ACTIVATION LAB**  
> `z = Wx+b → a=f(z)` ikusteko: Identity, Sigmoid, Tanh, ReLU eta Softmax.  
> **https://rbarrenago.github.io/5072-ml/demos/activation-lab/**


## 📝 Hiru modu

- 🟢 **KOMENTATU** · 🟡 **HUTSUNEAK BETE** (`____`) · 🔴 **KODETU ZEROTIK**

> ⚠️ Colab-en: *Exekuzio-ingurunea → GPU* (ez da ezinbestekoa saio honetan).


## 0. Inportazioak 🟢


In [ ]:
import numpy as np
import warnings; warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)  # erreproduzigarritasuna
print('TF bertsioa:', tf.__version__)

## 2.1 — XOR, orain Keras-en 🟢

S1eko ideia bera, baina **Keras-ekin**:

| S1 (eskuz) | Keras |
|---|---|
| `W1, b1, W2, b2` | `layers.Dense(...)` |
| `sigmoide(z)` / ReLU | `activation=...` |
| forward + gradienteak | `fit()` + autodiff/backprop |
| pisuen eguneraketa | `optimizer='adam'` |
| MSE / BCE | `loss=...` |

> **Backpropagation-ek gradienteak kalkulatzen ditu; optimizatzaileak (Adam/SGD) gradiente horiek erabiltzen ditu pisuak eguneratzeko.**


In [ ]:
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)  # XOR sarrerak: 4 adibide, 2 ezaugarri
y = np.array([0,1,1,0], dtype=float)  # XOR irteerak (1 sarrerak desberdinak direnean)

sarea = keras.Sequential([  # geruzak ordenan pilatzen dituen eredua
    layers.Input((2,)),  # sarrera-forma: 2 ezaugarri
    layers.Dense(4, activation='relu'),  # ezkutuko geruza: z=W1x+b1 (2x4) + ReLU → ez-lineala, XOR ebazteko beharrezkoa
    layers.Dense(1, activation='sigmoid')  # irteera: neurona 1 + sigmoide → P(y=1) [0,1] tartean
])
sarea.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])  # BCE galera (sailkapen bitarra) + Adam optimizatzailea + accuracy metrika
sarea.fit(X, y, epochs=2000, verbose=0)  # 2000 aldiz: forward → loss → backprop → pisuen eguneraketa
print('Iragarpenak:', np.round(sarea.predict(X, verbose=0).ravel(), 3))  # probabilitateak (≈0 edo ≈1 izan beharko lirateke)
print('Accuracy:', sarea.evaluate(X, y, verbose=0)[1])  # evaluate → [loss, accuracy]; [1] = accuracy (1.0 = 4/4 ondo)

### a) Sarearen egitura ikusi 🟢


In [ ]:
sarea.summary()  # geruzak eta parametro kopurua (pisuak + biasak)

### b) `summary()` ez da magia: parametroak eskuz 🟢

XOR sarean:

- `2 → 4`: `2×4` pisu + `4` bias = **12 parametro**
- `4 → 1`: `4×1` pisu + `1` bias = **5 parametro**
- Guztira = **17 parametro**

Keras-ek ikasten dituen matrize eta bektoreen formak ikus ditzagun.


In [ ]:
for p in sarea.trainable_weights:
    print(f'{p.name:25s} {tuple(p.shape)}')
print('Parametroak guztira:', sarea.count_params())


## 2.2 — Lehen sailkatzaile erreala: digituak 🟡

`scikit-learn`-eko digitu eskuz idatziak (8×8 = 64 pixel, **10 klase**: 0–9). Irteeran **10 neurona** eta **softmax** (klase bakoitzaren probabilitatea).

**Hutsuneak bete:** irteera-aktibazioa eta galera.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digituak = load_digits()
X = digituak.data  # (1797, 64): irudi bakoitza 8x8 pixel laututa
y = digituak.target  # etiketak: 0–9 zenbaki osoak
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler().fit(X_train)  # batezbestekoa/desbiderapena TRAIN-ekin bakarrik (datu-ihesa ekiditeko)
X_train_s = sc.transform(X_train); X_test_s = sc.transform(X_test)
print('Datuak:', X.shape, '| klaseak:', len(np.unique(y)), '| train/test:', len(X_train), len(X_test))  # 80/20 banaketa, stratify → klase-proportzio berdinak

In [ ]:
modeloa = keras.Sequential([
    layers.Input((64,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')  # softmax: 10 logit → 10 probabilitate (Σp=1)
])
# sparse_categorical_crossentropy: klase anitzak + etiketak zenbaki osoak (ez one-hot)
modeloa.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
modeloa.fit(X_train_s, y_train, epochs=30, verbose=0)
print('Test accuracy:', round(modeloa.evaluate(X_test_s, y_test, verbose=0)[1], 3))

### a) Iragarri digitu bat 🟢


In [ ]:
probak = modeloa.predict(X_test_s[:1], verbose=0)[0]
print('Benetakoa:', y_test[0])
print('Softmax probabilitateak:')
for digitua, p in enumerate(probak):
    print(f'  {digitua}: {p:.4f}')
print('Batura:', round(probak.sum(), 6))
print('argmax → iragarritako digitua:', int(np.argmax(probak)))
print('Konfiantza:', round(float(probak.max()), 3))


## 🧠 Gako-mezua

- `Sequential` = geruzak pilatu.
- `Dense(n, activation=...)` = `z = Wx+b` + aktibazio-funtzioa.
- **Backprop/autodiff** = gradienteak kalkulatu.
- `optimizer='adam'` = gradiente horiekin pisuak eguneratzeko estrategia.
- `loss=...` = zer minimizatu behar duen definitu.
- `fit()` = forward → loss → backprop → optimizer/update zikloa automatizatu.
- **Softmax** = logit-ak probabilitate-bektore bihurtu (`Σp = 1`); `argmax`-ek klasea aukeratzen du.

**Hurrengoan (S3):** kurbak irakurri, gaininskatzea diagnostikatu eta erregularizatu.


## Ondorioak

- Keras-ek S1eko kalkuluak automatizatzen ditu, baina oinarria bera da.
- `summary()`-ko parametroak pisu-matrize eta bias-bektore errealak dira.
- Klase anitzeko sailkapenean: **Softmax + Sparse Categorical Cross-Entropy**.
- Test accuracy-a exekuzioaren arabera zertxobait alda daiteke; garrantzitsuena pipeline-a eta interpretazioa dira.
- Hurrengoa: **S3 — entrenamendua eta erregularizazioa**.
